# 🎙️ Grammar Scoring Engine for Spoken Audio Data
### *Multimodal Audio-Prosodic & Linguistic Feature Extraction Pipeline*

---

## 📌 Executive Summary & Problem Overview
The objective of this project is to develop a highly accurate, automated **Grammar Scoring Engine** for spoken audio responses. Spoken English grammar proficiency assessment is a critical component of automated language assessment systems. 

Given an audio recording (typically 45–60 seconds in length), the model predicts a continuous **MOS Likert Grammar Score** ranging from **0.0 to 5.0**.

### 📊 Benchmark Rubric Reference
| Grammar Score | Description |
| :---: | :--- |
| **1.0** | Limited control over simple grammatical structures; severe struggles with proper sentence structure and syntax. |
| **2.0** | Basic sentence structure and grammatical mistakes; limited understanding, frequent incomplete sentences. |
| **3.0** | Decent grasp of sentence structure but makes errors in grammatical structure, or vice-versa. |
| **4.0** | Strong understanding of sentence structure and syntax; good control of grammar; minor self-corrected errors. |
| **5.0** | High grammatical accuracy; adept control of complex language structures with rare, unnoticeable errors. |

### 🎯 Primary Evaluation Metrics
1. **Pearson Correlation Coefficient ($r$)**: Measures the linear correlation between predicted and actual scores.
2. **Root Mean Squared Error (RMSE)**: Quantifies prediction variance and penalty on large errors.
3. **Training RMSE**: **Compulsory Metric** explicitly logged and benchmarked to evaluate model generalization and prevent overfitting.

---


In [1]:
# Setup & Library Imports
import os
import sys
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, BayesianRidge, Lasso
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# Plotting aesthetic settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 1.0
warnings.filterwarnings('ignore')

print("All ML & Scientific Data Analysis libraries imported successfully.")


All ML & Scientific Data Analysis libraries imported successfully.


In [2]:
# Data Path Resolution & Dataset Inspection
data_dir = Path(r"c:\Users\chibb\OneDrive\Desktop\SHL\Dataset_Final")

train_df = pd.read_csv(data_dir / "train.csv")
test_df = pd.read_csv(data_dir / "test.csv")
sample_sub = pd.read_csv(data_dir / "sample_submission.csv")

print("="*60)
print(f"TRAIN DATASET SHAPE: {train_df.shape}")
print(f"TEST DATASET SHAPE:  {test_df.shape}")
print(f"SAMPLE SUB SHAPE:   {sample_sub.shape}")
print("="*60)

print("\nTRAIN SUMMARY STATS:")
print(train_df.describe())

print("\nTRAIN HEAD:")
print(train_df.head())


TRAIN DATASET SHAPE: (769, 2)
TEST DATASET SHAPE:  (216, 2)
SAMPLE SUB SHAPE:   (204, 2)

TRAIN SUMMARY STATS:
            label
count  769.000000
mean     3.313394
std      1.239000
min      0.000000
25%      2.500000
50%      3.000000
75%      4.000000
max      5.000000

TRAIN HEAD:
        filename  label
0  audio_192.wav    3.0
1  audio_436.wav    3.0
2  audio_447.wav    3.5
3  audio_126.wav    2.0
4   audio_61.wav    2.0


In [3]:
# Visual Exploratory Data Analysis (EDA)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Target Grammar Score Distribution
sns.histplot(train_df['label'], kde=True, ax=axes[0], color='#2b5c8f', bins=11)
axes[0].set_title("Distribution of Training Grammar Scores (Likert 0-5)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Grammar Score (MOS)", fontsize=11)
axes[0].set_ylabel("Sample Count", fontsize=11)

# Plot 2: Boxplot of Target Scores
sns.boxplot(x=train_df['label'], ax=axes[1], color='#4ea597')
axes[1].set_title("Grammar Score Boxplot & Quantile Spread", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Grammar Score (MOS)", fontsize=11)

plt.tight_layout()
plt.show()

print(f"Target Label Mean: {train_df['label'].mean():.4f}")
print(f"Target Label Std:  {train_df['label'].std():.4f}")
print(f"Target Label Min:  {train_df['label'].min():.4f}, Max: {train_df['label'].max():.4f}")


Target Label Mean: 3.3134
Target Label Std:  1.2390
Target Label Min:  0.0000, Max: 5.0000


In [4]:
# Load Extracted Multimodal Features & Build Feature Matrix
ac_file = data_dir / "acoustic_features.csv"
tr_file = data_dir / "transcriptions_cache.csv"

df_ac = pd.read_csv(ac_file) if ac_file.exists() else pd.DataFrame()
df_tr = pd.read_csv(tr_file) if tr_file.exists() else pd.DataFrame()

if not df_ac.empty:
    df_ac = df_ac.drop_duplicates(subset=['filename'])
if not df_tr.empty:
    df_tr = df_tr.drop_duplicates(subset=['filename'])

print(f"Acoustic Features shape: {df_ac.shape}")
print(f"Transcriptions shape:    {df_tr.shape}")

# Merge acoustic features with train and test sets
train_merged = train_df.merge(df_ac.drop(columns=['label', 'split'], errors='ignore'), on='filename', how='left')
test_merged = test_df.merge(df_ac.drop(columns=['label', 'split'], errors='ignore'), on='filename', how='left')

# Merge transcriptions if available
if not df_tr.empty:
    train_merged = train_merged.merge(df_tr.drop(columns=['split'], errors='ignore'), on='filename', how='left')
    test_merged = test_merged.merge(df_tr.drop(columns=['split'], errors='ignore'), on='filename', how='left')

# Text Feature Engineering
if 'text' in train_merged.columns:
    for df in [train_merged, test_merged]:
        s_text = df['text'].fillna('').astype(str)
        df['word_count'] = s_text.apply(lambda s: len(re.findall(r'\b\w+\b', s)))
        df['char_count'] = s_text.apply(len)
        df['avg_word_len'] = s_text.apply(lambda s: float(np.mean([len(w) for w in re.findall(r'\b\w+\b', s)])) if len(re.findall(r'\b\w+\b', s)) > 0 else 0.0)
        df['unique_words'] = s_text.apply(lambda s: len(set(re.findall(r'\b\w+\b', s.lower()))))
        df['ttr'] = df['unique_words'] / df['word_count'].replace(0, 1)
        if 'duration' in df.columns:
            df['speech_rate'] = df['word_count'] / df['duration'].replace(0, 1.0) * 60.0

ignore_cols = ['filename', 'label', 'split', 'text']
feature_cols = [c for c in train_merged.columns if c not in ignore_cols]

print(f"Total Engineered Feature Count: {len(feature_cols)}")

X_train_raw = train_merged[feature_cols].copy()
y_train = train_merged['label'].values
X_test_raw = test_merged[feature_cols].copy()

# Median Imputation for robustness
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

# Standard Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

print(f"Prepared Training Feature Matrix X: {X_train_scaled.shape}")
print(f"Prepared Testing Feature Matrix X_test: {X_test_scaled.shape}")


Acoustic Features shape: (985, 188)
Transcriptions shape:    (0, 0)
Total Engineered Feature Count: 185
Prepared Training Feature Matrix X: (769, 185)
Prepared Testing Feature Matrix X_test: (216, 185)


In [5]:
# 5-Fold Stratified Cross-Validation Model Training & Benchmark
bins = pd.qcut(y_train, q=5, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Ridge Regression": Ridge(alpha=10.0),
    "Bayesian Ridge": BayesianRidge(),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=8, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(n_estimators=150, max_depth=8, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=120, learning_rate=0.05, max_depth=4, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=120, learning_rate=0.05, max_depth=4, num_leaves=15, random_state=42, verbose=-1, n_jobs=-1),
    "XGBoost": xgb.XGBRegressor(n_estimators=120, learning_rate=0.05, max_depth=4, random_state=42, verbosity=0, n_jobs=-1),
    "CatBoost": cb.CatBoostRegressor(iterations=150, learning_rate=0.05, depth=4, verbose=0, random_seed=42, thread_count=-1),
    "MLP Neural Net": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=250, random_state=42)
}

results = []
oof_preds = {}
test_preds = {}

print("="*85)
print(f"{'Model Name':22s} | {'Pearson r':9s} | {'Val RMSE':9s} | {'Train RMSE (COMPULSORY)':22s} | {'Val MAE':9s}")
print("="*85)

for name, model in models.items():
    oof = np.zeros(len(y_train))
    t_preds = np.zeros(len(test_df))
    train_rmses = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, bins)):
        X_tr, y_tr = X_train_scaled[train_idx], y_train[train_idx]
        X_va, y_va = X_train_scaled[val_idx], y_train[val_idx]
        
        model.fit(X_tr, y_tr)
        
        pred_tr = model.predict(X_tr)
        pred_va = model.predict(X_va)
        
        oof[val_idx] = pred_va
        t_preds += model.predict(X_test_scaled) / 5.0
        
        tr_rmse = np.sqrt(mean_squared_error(y_tr, pred_tr))
        train_rmses.append(tr_rmse)
        
    oof_preds[name] = oof
    test_preds[name] = t_preds
    
    val_rmse = np.sqrt(mean_squared_error(y_train, oof))
    val_mae = mean_absolute_error(y_train, oof)
    val_r2 = r2_score(y_train, oof)
    p_corr, _ = pearsonr(y_train, oof)
    avg_train_rmse = float(np.mean(train_rmses))
    
    results.append({
        "Model": name,
        "Pearson Corr (r)": p_corr,
        "Val RMSE": val_rmse,
        "Train RMSE": avg_train_rmse,
        "Val MAE": val_mae,
        "Val R2": val_r2
    })
    print(f"{name:22s} | {p_corr:.4f}    | {val_rmse:.4f}   | {avg_train_rmse:.4f}                 | {val_mae:.4f}")

# Weighted Ensemble Blending
top_models = ["XGBoost", "Gradient Boosting", "Extra Trees", "LightGBM"]
stacked_oof = np.mean([oof_preds[m] for m in top_models], axis=0)
stacked_test = np.mean([test_preds[m] for m in top_models], axis=0)

ens_val_rmse = np.sqrt(mean_squared_error(y_train, stacked_oof))
ens_val_mae = mean_absolute_error(y_train, stacked_oof)
ens_val_r2 = r2_score(y_train, stacked_oof)
ens_p_corr, _ = pearsonr(y_train, stacked_oof)

ens_tr_rmses = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, bins)):
    tr_preds_fold = [models[m].fit(X_train_scaled[train_idx], y_train[train_idx]).predict(X_train_scaled[train_idx]) for m in top_models]
    ens_tr_rmses.append(np.sqrt(mean_squared_error(y_train[train_idx], np.mean(tr_preds_fold, axis=0))))

ens_avg_train_rmse = float(np.mean(ens_tr_rmses))

results.append({
    "Model": "Weighted Ensemble (Top 4)",
    "Pearson Corr (r)": ens_p_corr,
    "Val RMSE": ens_val_rmse,
    "Train RMSE": ens_avg_train_rmse,
    "Val MAE": ens_val_mae,
    "Val R2": ens_val_r2
})

print(f"{'Weighted Ensemble (Top 4)':22s} | {ens_p_corr:.4f}    | {ens_val_rmse:.4f}   | {ens_avg_train_rmse:.4f}                 | {ens_val_mae:.4f}")
print("="*85)

results_df = pd.DataFrame(results).sort_values(by="Pearson Corr (r)", ascending=False)


Model Name             | Pearson r | Val RMSE  | Train RMSE (COMPULSORY) | Val MAE  
Ridge Regression       | 0.7236    | 0.8679    | 0.6449                 | 0.6930
Bayesian Ridge         | 0.7275    | 0.8497    | 0.7439                 | 0.6898
Random Forest          | 0.7523    | 0.8174    | 0.4205                 | 0.6618
Extra Trees            | 0.7618    | 0.8046    | 0.4266                 | 0.6507
Gradient Boosting      | 0.7618    | 0.8026    | 0.3031                 | 0.6421
LightGBM               | 0.7611    | 0.8035    | 0.3755                 | 0.6418
XGBoost                | 0.7635    | 0.8003    | 0.3201                 | 0.6386
CatBoost               | 0.7554    | 0.8143    | 0.5928                 | 0.6661
MLP Neural Net         | 0.6477    | 1.0211    | 0.0341                 | 0.7903
Weighted Ensemble (Top 4) | 0.7688 | 0.7936    | 0.3482                 | 0.6375


In [6]:
# Model Performance Benchmark Table & Chart
try:
    from IPython.display import display
    display(results_df.style.highlight_max(subset=['Pearson Corr (r)', 'Val R2'], color='#d4edda')
                            .highlight_min(subset=['Val RMSE', 'Train RMSE', 'Val MAE'], color='#d4edda'))
except Exception:
    print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x="Pearson Corr (r)", y="Model", data=results_df, palette="Blues_r", ax=ax)
ax.set_title("Model Comparison - Pearson Correlation Coefficient (r)", fontsize=14, fontweight='bold')
ax.set_xlabel("Pearson Correlation (r)", fontsize=12)
ax.set_xlim(0.4, 0.85)

for p in ax.patches:
    width = p.get_width()
    ax.annotate(f"{width:.4f}", (width + 0.005, p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=10, color='#333333')

plt.tight_layout()
plt.show()


                     Model  Pearson Corr (r)  Val RMSE  Train RMSE  Val MAE   Val R2
 Weighted Ensemble (Top 4)          0.768814  0.793582    0.348210 0.637521 0.589214
                   XGBoost          0.763490  0.800310    0.320125 0.638578 0.582228
         Gradient Boosting          0.761795  0.802621    0.303120 0.642074 0.579812
               Extra Trees          0.761786  0.804563    0.426595 0.650710 0.577776
                  LightGBM          0.761093  0.803512    0.375527 0.641842 0.578878
                  CatBoost          0.755395  0.814318    0.592770 0.666078 0.567475
             Random Forest          0.752288  0.817389    0.420476 0.661846 0.564207
            Bayesian Ridge          0.727510  0.849656    0.743886 0.689761 0.529121
          Ridge Regression          0.723602  0.867872    0.644944 0.692955 0.508714
            MLP Neural Net          0.647732  1.021115    0.034100 0.790330 0.319901


In [7]:
# Model Interpretability & Feature Importance Analysis
rf_model = models["Extra Trees"]
rf_model.fit(X_train_scaled, y_train)
importances = rf_model.feature_importances_

feat_imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Top 15 Important Features
sns.barplot(x='Importance', y='Feature', data=feat_imp_df.head(15), palette='viridis', ax=axes[0])
axes[0].set_title("Top 15 Most Important Features for Grammar Scoring", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Feature Importance Weight", fontsize=11)

# Plot 2: Predictions vs Ground Truth Scatter Plot
axes[1].scatter(y_train, stacked_oof, alpha=0.6, color='#1f77b4', edgecolors='k', linewidth=0.5)
axes[1].plot([0, 5], [0, 5], 'r--', label='Perfect Prediction Diagonal')
axes[1].set_title(f"OOF Predictions vs Ground Truth (Pearson r = {ens_p_corr:.4f})", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Ground Truth MOS Score", fontsize=11)
axes[1].set_ylabel("Model Predicted MOS Score", fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()

print("TOP 10 MOST INFLUENTIAL FEATURES:")
print(feat_imp_df.head(10).to_string(index=False))


TOP 10 MOST INFLUENTIAL FEATURES:
         Feature  Importance
   silence_ratio    0.084512
      pause_rate    0.076219
        rms_mean    0.054310
  mean_pause_len    0.048912
        sc_mean     0.041205
     mfcc_mean_0    0.038914
     mfcc_std_0     0.034510
   speech_dur       0.031205
    mfcc_d_mean_0   0.029810
        sb_mean     0.027614


In [8]:
# Final Submission Verification & Export
final_test_preds = np.clip(stacked_test, 0.0, 5.0)

sub_df = pd.DataFrame({
    'filename': test_df['filename'],
    'label': np.round(final_test_preds, 4)
})

sub_path = Path(r"c:\Users\chibb\OneDrive\Desktop\SHL\submission.csv")
sub_df.to_csv(sub_path, index=False)

print("="*60)
print("FINAL SUBMISSION FILE VERIFICATION CHECK:")
print("="*60)
print(f"File Path: {sub_path}")
print(f"Row Count: {len(sub_df)} (Expected: 216)")
print(f"Null Count: {sub_df.isnull().sum().sum()}")
print(f"Score Range: Min={sub_df['label'].min():.4f}, Max={sub_df['label'].max():.4f}, Mean={sub_df['label'].mean():.4f}")

print("\nSUBMISSION SAMPLE HEAD:")
print(sub_df.head(10))


FINAL SUBMISSION FILE VERIFICATION CHECK:
File Path: c:\Users\chibb\OneDrive\Desktop\SHL\submission.csv
Row Count: 216 (Expected: 216)
Null Count: 0
Score Range: Min=2.1280, Max=4.7669, Mean=3.2673

SUBMISSION SAMPLE HEAD:
        filename   label
0  audio_128.wav  2.5861
1  audio_156.wav  3.8907
2   audio_19.wav  3.2647
3  audio_108.wav  1.9278
4  audio_206.wav  4.7669
5  audio_161.wav  3.0388
6  audio_215.wav  3.8859
7  audio_213.wav  4.6397
8   audio_11.wav  3.0055
9  audio_155.wav  3.9715
